# Sobel CUDA no Google Colab

Este notebook executa a versão CUDA do projeto em uma GPU do Colab. Antes de rodar as células, selecione **Ambiente de execução > Alterar o tipo de ambiente de execução > Acelerador de hardware: GPU**. Se a opção estiver disponível, escolha uma **T4**.

O fluxo é: verificar a GPU, gravar os arquivos C/CUDA, compilar, enviar uma imagem (PNG, JPG, JPEG, TIFF ou PGM), executar o Sobel e baixar o PNG resultante.

In [ ]:
# Deve mostrar uma GPU NVIDIA.
!nvidia-smi

## Arquivos do projeto

As duas células seguintes recriam, dentro do ambiente temporário do Colab, o cabeçalho PGM e o arquivo `.cu`.

In [ ]:
%%writefile pgm.h
#ifndef SOBEL_PGM_H
#define SOBEL_PGM_H

#include <ctype.h>
#include <errno.h>
#include <limits.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

typedef struct {
    int largura;
    int altura;
    unsigned char *pixels;
} ImagemPGM;

static int proximo_token(FILE *f, char *token, size_t capacidade) {
    int c;
    size_t i = 0;
    do {
        c = fgetc(f);
        if (c == '#')
            do { c = fgetc(f); } while (c != '\n' && c != EOF);
    } while (c != EOF && isspace((unsigned char)c));

    if (c == EOF) return 0;
    while (c != EOF && !isspace((unsigned char)c) && c != '#') {
        if (i + 1 >= capacidade) return 0;
        token[i++] = (char)c;
        c = fgetc(f);
    }
    token[i] = '\0';
    if (c == '#')
        do { c = fgetc(f); } while (c != '\n' && c != EOF);
    else if (c == '\r') {
        int seguinte = fgetc(f);
        if (seguinte != '\n' && seguinte != EOF) ungetc(seguinte, f);
    }
    return i != 0;
}

static int ler_inteiro_positivo(const char *s, int *valor) {
    char *fim;
    long n;
    errno = 0;
    n = strtol(s, &fim, 10);
    if (errno || *fim || n <= 0 || n > INT_MAX) return 0;
    *valor = (int)n;
    return 1;
}

static int ler_pgm(const char *caminho, ImagemPGM *img) {
    FILE *f = fopen(caminho, "rb");
    char token[64];
    int maxvalor;
    size_t n;

    img->largura = img->altura = 0;
    img->pixels = NULL;
    if (!f) { perror(caminho); return 0; }
    if (!proximo_token(f, token, sizeof(token)) || strcmp(token, "P5") != 0 ||
        !proximo_token(f, token, sizeof(token)) ||
        !ler_inteiro_positivo(token, &img->largura) ||
        !proximo_token(f, token, sizeof(token)) ||
        !ler_inteiro_positivo(token, &img->altura) ||
        !proximo_token(f, token, sizeof(token)) ||
        !ler_inteiro_positivo(token, &maxvalor) || maxvalor != 255) {
        fprintf(stderr, "Formato invalido: esperado PGM P5 com maxvalor 255: %s\n", caminho);
        fclose(f);
        return 0;
    }
    if ((size_t)img->altura > SIZE_MAX / (size_t)img->largura) {
        fprintf(stderr, "Dimensoes grandes demais.\n");
        fclose(f);
        return 0;
    }
    n = (size_t)img->largura * (size_t)img->altura;
    img->pixels = (unsigned char *)malloc(n);
    if (!img->pixels) {
        fprintf(stderr, "Falha ao alocar %zu bytes.\n", n);
        fclose(f);
        return 0;
    }
    if (fread(img->pixels, 1, n, f) != n) {
        fprintf(stderr, "Arquivo PGM truncado: %s\n", caminho);
        free(img->pixels);
        img->pixels = NULL;
        fclose(f);
        return 0;
    }
    fclose(f);
    return 1;
}

static int escrever_pgm(const char *caminho, const ImagemPGM *img) {
    FILE *f = fopen(caminho, "wb");
    size_t n = (size_t)img->largura * (size_t)img->altura;
    int ok;
    if (!f) { perror(caminho); return 0; }
    ok = fprintf(f, "P5\n%d %d\n255\n", img->largura, img->altura) > 0 &&
         fwrite(img->pixels, 1, n, f) == n;
    if (fclose(f) != 0) ok = 0;
    if (!ok) fprintf(stderr, "Falha ao gravar: %s\n", caminho);
    return ok;
}

static int ler_repeticoes(const char *s, int *valor) {
    if (!ler_inteiro_positivo(s, valor) || *valor > 1000000) {
        fprintf(stderr, "Repeticoes deve ser um inteiro entre 1 e 1000000.\n");
        return 0;
    }
    return 1;
}

#endif


In [ ]:
%%writefile sobel_cuda.cu
// Filtro de Sobel em CUDA C++ (PGM P5, tons de cinza)
#define _POSIX_C_SOURCE 200809L
#include <cuda_runtime.h>
#include <math.h>
#include <time.h>
#include "pgm.h"

static double agora_segundos(void) {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return (double)t.tv_sec + (double)t.tv_nsec * 1e-9;
}

// CUDA 1: cada thread executa o calculo de um pixel.
__global__ void sobel_cuda(const unsigned char *entrada, unsigned char *saida,
                           int largura, int altura) {
    // CUDA 2: indice global da thread, conforme visto em aula.
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int n = largura * altura;
    if (i >= n) return;

    int y = i / largura;
    int x = i % largura;
    if (x == 0 || y == 0 || x == largura - 1 || y == altura - 1) {
        saida[i] = 0;
        return;
    }

    int a = entrada[(y - 1) * largura + (x - 1)];
    int b = entrada[(y - 1) * largura + x];
    int c = entrada[(y - 1) * largura + (x + 1)];
    int d = entrada[y * largura + (x - 1)];
    int f = entrada[y * largura + (x + 1)];
    int g = entrada[(y + 1) * largura + (x - 1)];
    int h = entrada[(y + 1) * largura + x];
    int j = entrada[(y + 1) * largura + (x + 1)];

    int gx = -a + c - 2 * d + 2 * f - g + j;
    int gy = -a - 2 * b - c + g + 2 * h + j;
    float magnitude = sqrtf((float)(gx * gx + gy * gy));
    if (magnitude > 255.0f) magnitude = 255.0f;
    saida[i] = (unsigned char)(magnitude + 0.5f);
}

int main(int argc, char **argv) {
    ImagemPGM entrada = {0, 0, NULL};
    ImagemPGM saida = {0, 0, NULL};
    int repeticoes = 10;
    size_t n;
    double inicio, fim;
    int ok;
    unsigned char *entrada_host;
    size_t threads_por_bloco = 256;
    size_t numero_de_blocos;

    if (argc < 3 || argc > 4) {
        fprintf(stderr, "Uso: %s entrada.pgm saida.pgm [repeticoes]\n", argv[0]);
        return EXIT_FAILURE;
    }
    if (argc == 4 && !ler_repeticoes(argv[3], &repeticoes)) return EXIT_FAILURE;
    if (!ler_pgm(argv[1], &entrada)) return EXIT_FAILURE;

    n = (size_t)entrada.largura * (size_t)entrada.altura;
    saida.largura = entrada.largura;
    saida.altura = entrada.altura;

    // CUDA 3: memoria unificada, acessivel pela CPU e pela GPU.
    entrada_host = entrada.pixels;
    cudaMallocManaged((void **)&entrada.pixels, n);
    memcpy(entrada.pixels, entrada_host, n);
    free(entrada_host);
    cudaMallocManaged((void **)&saida.pixels, n);

    numero_de_blocos = (n + threads_por_bloco - 1) / threads_por_bloco;

    // Aquecimento fora da regiao temporizada.
    sobel_cuda<<<numero_de_blocos, threads_por_bloco>>>(
        entrada.pixels, saida.pixels, entrada.largura, entrada.altura);
    cudaDeviceSynchronize();

    inicio = agora_segundos();
    for (int i = 0; i < repeticoes; ++i) {
        sobel_cuda<<<numero_de_blocos, threads_por_bloco>>>(
            entrada.pixels, saida.pixels, entrada.largura, entrada.altura);
        cudaDeviceSynchronize();
    }
    fim = agora_segundos();

    printf("Imagem: %d x %d pixels\n", entrada.largura, entrada.altura);
    printf("Repeticoes: %d\n", repeticoes);
    printf("Tempo medio GPU (kernel + sincronizacao por repeticao): %.6f ms\n",
           1000.0 * (fim - inicio) / repeticoes);

    ok = escrever_pgm(argv[2], &saida);
    cudaFree(saida.pixels);
    cudaFree(entrada.pixels);
    return ok ? EXIT_SUCCESS : EXIT_FAILURE;
}


In [ ]:
# O Colab ja inclui o nvcc no ambiente com GPU.
!nvcc -O3 sobel_cuda.cu -o sobel_cuda

## Enviar a imagem

Execute a célula e selecione uma imagem do seu computador. Ela será convertida para PGM P5 em tons de cinza, formato usado pelo programa CUDA.

In [ ]:
from google.colab import files
from pathlib import Path
from PIL import Image

arquivos = files.upload()
if len(arquivos) != 1:
    raise ValueError("Envie exatamente uma imagem por vez.")

entrada_original = Path(next(iter(arquivos)))
with Image.open(entrada_original) as imagem:
    imagem_cinza = imagem.convert("L")
    largura, altura = imagem_cinza.size
    Path("entrada.pgm").write_bytes(
        f"P5\n{largura} {altura}\n255\n".encode("ascii") + imagem_cinza.tobytes()
    )

print(f"Entrada: {entrada_original.name} ({largura} x {altura} pixels)")

In [ ]:
import subprocess

REPETICOES = 10  # Altere se quiser uma medicao com mais execucoes.
processo = subprocess.run(
    ["./sobel_cuda", "entrada.pgm", "saida.pgm", str(REPETICOES)],
    check=True, text=True, capture_output=True
)
print(processo.stdout, end="")

In [ ]:
from IPython.display import display

with Image.open("saida.pgm") as resultado:
    resultado_png = resultado.convert("L")
    resultado_png.save("saida_sobel.png")
    display(resultado_png)

files.download("saida_sobel.png")